# Extended Hamiltonian Simulation

This exercise simulates a situation in which we want to couple our exisitng workflow with an exisiting C++ library.
We also want to extend the capabilities to levarage parallelism once we move to an HPC system.

**Learning outcomse**
1. Integrating existing codes in tierkreis
2. Extending workflows with parallelism.

In this setting we have a dummy C++ binary, which is already implemented as a worker.
## Task 1: The worker
Familiarize yourself with the worker api and how the worker functions.
See [api.tsp](../workers/cpp_worker/api.tsp) and [main.cpp](../workers/cpp_worker/main.cpp).
*Optional:* If you want you can implement custom logic to generate values for you.

## Task 2: Extend the existing graph

Extend the exisiting graph using the `embed` operation.


In [ ]:
from typing import NamedTuple

from cpp_worker import gen_inputs, Ansatz
from tierkreis.builder import Graph
from tierkreis.models import OpaqueType, TKR, Workflow
from graphs.guppy_main import hamiltonian_sim , SymbolicExecutionInputs

type Hamiltonian = list[tuple[OpaqueType["pytket._tket.pauli.QubitPauliString"], float]]

class ParallelExecutionInputs(NamedTuple):
    value: TKR[str]
    ham: TKR[Hamiltonian]
    ansatz: TKR[OpaqueType["hugr.package.Package"]]

def parallel_sim() -> Workflow[ParallelExecutionInputs, TKR[float]]:
    simulation_graph = Graph(ParallelExecutionInputs, TKR[float])
    ansatz_inputs: Ansatz = simulation_graph.task(
        gen_inputs(simulation_graph.inputs.value)
    )

    embedded = simulation_graph.embed(
        hamiltonian_sim(),
        SymbolicExecutionInputs(ansatz_inputs.a, ansatz_inputs.b, ansatz_inputs.c, simulation_graph.inputs.ham, simulation_graph.inputs.ansatz),
        TKR[float],
    )

    return simulation_graph.finish_with_outputs(embedded)

### Task 3: Map over multiple hamiltonians

Use the above graph in a map.


*Note:* For simplicity you can map over the entire graph here.
For a real world scenarion where your C++ worker has potentially long running times you would rewrite the graph to have the task outside the map and do it only once.

In [ ]:
class MultipleHamiltoniansInputs(NamedTuple):
    value: TKR[str]
    hamiltonians: ... # TODO: add the new input here
    ansatz: TKR[OpaqueType["hugr.package.Package"]]


def multiple_hamiltonian_sim() -> Workflow[
    MultipleHamiltoniansInputs, TKR[list[float]]
]:
    simulation_graph = Graph(MultipleHamiltoniansInputs, TKR[list[float]])
    map_inputs = simulation_graph.map(
        lambda x: ParallelExecutionInputs(
            simulation_graph.inputs.value, x, simulation_graph.inputs.ansatz
        ),
        simulation_graph.inputs.hamiltonians,
    )
    exp_values = simulation_graph.map(parallel_sim(), map_inputs)

    return simulation_graph.finish_with_outputs(exp_values)

We now define our inputs as before:

In [ ]:
from typing import Any

from guppylang import guppy
from guppylang.std.angles import angle
from guppylang.std.builtins import array
from guppylang.std.quantum import cx, qubit, rz
from pytket import Qubit
from pytket.pauli import Pauli, QubitPauliString

@guppy(link_name="ansatz")
def ansatz(qbts: array[qubit, 4], a: float, b: float, c: float) -> None:  # type: ignore
    for i in array(0, 1, 2):
        cx(qbts[i], qbts[i + 1])
    rz(qbts[0], angle(a))
    for i in array(2, 1, 0):
        cx(qbts[i], qbts[i + 1])
    rz(qbts[0], angle(b))
    for i in array(0, 1, 2):
        cx(qbts[i], qbts[i + 1])
    rz(qbts[0], angle(c))
    for i in array(2, 1, 0):
        cx(qbts[i], qbts[i + 1])

def inputs() -> dict[str, Any]:
    ansatz_package = guppy.library(ansatz).compile()
    qubits = [Qubit(0), Qubit(1), Qubit(2), Qubit(3)]
    hamiltonians = [
        [
            (
                QubitPauliString(
                    qubits, [Pauli.X, Pauli.Y, Pauli.X, Pauli.I]
                ).to_list(),
                0.1,
            ),
            (
                QubitPauliString(
                    qubits, [Pauli.Y, Pauli.Z, Pauli.X, Pauli.Z]
                ).to_list(),
                0.5,
            ),
            (
                QubitPauliString(
                    qubits, [Pauli.X, Pauli.Y, Pauli.Z, Pauli.I]
                ).to_list(),
                0.3,
            ),
            (
                QubitPauliString(
                    qubits, [Pauli.Z, Pauli.Y, Pauli.X, Pauli.Y]
                ).to_list(),
                0.6,
            ),
        ],
        [
            (
                QubitPauliString(
                    qubits, [Pauli.X, Pauli.Y, Pauli.X, Pauli.I]
                ).to_list(),
                0.2,
            ),
            (
                QubitPauliString(
                    qubits, [Pauli.Y, Pauli.Z, Pauli.X, Pauli.Z]
                ).to_list(),
                0.4,
            ),
            (
                QubitPauliString(
                    qubits, [Pauli.X, Pauli.Y, Pauli.Z, Pauli.I]
                ).to_list(),
                0.4,
            ),
            (
                QubitPauliString(
                    qubits, [Pauli.Z, Pauli.Y, Pauli.X, Pauli.Y]
                ).to_list(),
                0.5,
            ),
        ],
    ]
    return {
        "ansatz": ansatz_package,
        "hamiltonians": hamiltonians,
        "value": "JW",
    }

Finally we can run the graph:

In [ ]:
import logging
from uuid import UUID

from tierkreis.controller import run_graph
from tierkreis.executor import ShellExecutor
from tierkreis.storage import FileStorage, read_outputs


def main() -> None:
    graph = hamiltonian_sim()
    storage = FileStorage(
        workflow_id=UUID(int=12346), name="parallel_hamiltonian_simulation"
    )
    executor = ShellExecutor(None, storage.workflow_dir)
    storage.clean_graph_files()
    run_graph(storage, executor, graph, inputs())
    result = read_outputs(graph, storage)
    print("Value is: ", result)

logging.basicConfig(level=logging.INFO)
main()